In [ ]:
#Установка библиотек
!pip install -q transformers evaluate accelerate
!pip install -q --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00


In [ ]:

import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    AutoModel,
    AutoConfig
)
from transformers.modeling_outputs import SequenceClassifierOutput
import evaluate
from google.colab import userdata
from huggingface_hub import login
from torch.optim import AdamW

# Загрузка датасета RUSSE из RussianSuperGLUE
login(token=userdata.get('HF_TOKEN'))
model_name = "DeepPavlov/rubert-base-cased"

# Загрузка датасета из локальных jsonl файлов
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

tokenizer = AutoTokenizer.from_pretrained(model_name)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
class BertWithLAYA(nn.Module):
    def __init__(self, model_name="DeepPavlov/rubert-base-cased", num_labels=2):
        super().__init__()

        # Загрузка конфига модели
        config = AutoConfig.from_pretrained(model_name)

        # Загружаем базовую модель с иницилизацией весов
        self.bert = AutoModel.from_config(config)

        ## Заморозка всех весов BERT
        #for param in self.bert.parameters():
        #    param.requires_grad = False

        hidden_size = config.hidden_size

        # Модуль LAYA: Линейный слой, переводящий hidden_size в 1 скаляр для каждого слоя
        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)

        # Итоговый классификатор
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        # 1. Пропускаем данные через BERT, ОБЯЗАТЕЛЬНО запрашивая скрытые состояния
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True
        )

        # outputs.hidden_states — это кортеж из 13 тензоров (1 слой эмбеддингов + 12 слоев трансформера)
        # Берем только 12 слоев трансформера. Размерность каждого: (batch_size, seq_len, hidden_size)
        hidden_states = outputs.hidden_states[1:]

        # Собираем их в один тензор: (batch_size, num_layers=12, seq_len, hidden_size)
        stacked_hidden_states = torch.stack(hidden_states, dim=1)

        # Берем только [CLS] токены (они находятся на нулевой позиции в seq_len)
        # Размерность: (batch_size, 12, hidden_size)
        cls_states = stacked_hidden_states[:, :, 0, :]

        # 2. Механизм LAYA
        # Применяем линейный слой ко всем слоям: (batch_size, 12, 1)
        attention_scores = self.attention_weights(cls_states)

        # Пропускаем через Softmax по измерению слоев (dim=1)
        attention_probs = torch.softmax(attention_scores, dim=1)

        # Умножаем [CLS] каждого слоя на его вес и суммируем
        # (batch_size, 12, hidden_size) * (batch_size, 12, 1) -> sum(dim=1) -> (batch_size, hidden_size)
        aggregated_representation = torch.sum(attention_probs * cls_states, dim=1)

        # 3. Классификация
        logits = self.classifier(aggregated_representation)

        # 4. Расчет Loss (Требуется для Trainer)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels.view(-1))

        # Возвращаем стандартный формат HuggingFace, чтобы Trainer мог с ним работать
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None
        )

In [ ]:
# Функция препроцессинга с явным выделением целевого слова
def preprocess_function(examples):
    first_sentences = [
        f"Слово: {w}. Контекст 1: {s1}"
        for w, s1 in zip(examples["word"], examples["sentence1"])
    ]
    second_sentences = [
        f"Контекст 2: {s2}"
        for s2 in examples["sentence2"]
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=128,
        padding=False
    )

    # Безопасное извлечение меток
    raw_labels = examples.get("label", examples.get("state", []))

    # конвертация любых форматов
    tokenized["label"] = [1 if str(l).lower() in ['true', '1'] else 0 for l in raw_labels]

    return tokenized

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset = tokenized_dataset.cast_column("label", Value("int64"))

# Фиксация формата колонок
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "label"]
)



Map:   0%|          | 0/19845 [00:00<?, ? examples/s]

Map:   0%|          | 0/8505 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/19845 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8505 [00:00<?, ? examples/s]

In [ ]:
# Метрика Accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)



In [ ]:
model = BertWithLAYA(model_name="DeepPavlov/rubert-base-cased", num_labels=2)
model = model.to("cuda")
for param in model.parameters():
    param.data = param.data.contiguous()

bert_params = [p for n, p in model.named_parameters() if n.startswith("bert")]
laya_and_classifier_params = [p for n, p in model.named_parameters() if not n.startswith("bert")]

# Создаем оптимайзер с разными LR
optimizer = AdamW([
    {"params": bert_params, "lr": 1e-4},                 # Низкая скорость для трансформера
    {"params": laya_and_classifier_params, "lr": 5e-4}   # Высокая скорость для LAYA и классификатора
], weight_decay=0.01)

In [ ]:
# Гиперпараметры обучения
training_args = TrainingArguments(
    output_dir="./rubert_russe_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=100,
    weight_decay=0.01,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none"
)



In [16]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)  # <--- Передаем кастомный оптимайзер
)

# Старт обучения
print("Начинаем обучение...")
trainer.train()


Начинаем обучение...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.645338,0.632955,0.640564
2,0.552815,0.594423,0.695473
3,0.441387,0.585989,0.725220
4,0.303295,0.668167,0.727454
5,0.232469,0.877983,0.720870
6,0.178218,0.926844,0.729336
7,0.138880,1.326182,0.722869
8,0.116774,1.263255,0.712169


Epoch,Training Loss,Validation Loss,Accuracy
1,0.645338,0.632955,0.640564
2,0.552815,0.594423,0.695473
3,0.441387,0.585989,0.725220
4,0.303295,0.668167,0.727454
5,0.232469,0.877983,0.720870
6,0.178218,0.926844,0.729336
7,0.138880,1.326182,0.722869
8,0.116774,1.263255,0.712169


KeyboardInterrupt: 

In [ ]:

# Финальная оценка
eval_results = trainer.evaluate()
print("Validation Accuracy:", eval_results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.643235,0.634768,0.648207
2,0.638865,0.619473,0.665373
3,0.627364,0.627986,0.647501
4,0.612462,0.616154,0.661141
5,0.621887,0.614459,0.671487
6,0.616932,0.612963,0.673016
7,0.603245,0.610395,0.668665
8,0.619817,0.617936,0.666784
9,0.617839,0.606859,0.671840
10,0.613886,0.607360,0.675250


Validation Accuracy: 0.6752498530276309


In [ ]:
# Архивируем всю папку с результатами
!zip -r rubert_russe_results.zip /content/rubert_russe_results

# Запускаем скачивание архива
from google.colab import files
files.download('rubert_russe_results.zip')